# DBMS Lecture 3: Advanced Database Topics

## Recap of Lecture 2
In the previous lecture, we explored:
- **Logical Execution Order**: Understanding that SQL queries are not executed in the order they are written. Filtering data early (in the `WHERE` clause) is significantly more efficient than filtering after aggregation (in the `HAVING` clause).
- **Concurrency and Databases**: How multi-threading and connection pooling impact database performance and resource management.
- **Data Generation**: Using libraries like `Faker` to generate large volumes of mock data for performance testing.


## Comprehensive Recap: Database JOINs

Before comparing subqueries and `JOIN` operations, a review of the various types of `JOIN`s is essential. `JOIN` clauses are used to combine rows from two or more tables based on a related column between them.

### Types of JOINs:

- **INNER JOIN**: Retrieves records that have matching values in both tables.
- **LEFT (OUTER) JOIN**: Retrieves all records from the left table, and the matched records from the right table.
- **RIGHT (OUTER) JOIN**: Retrieves all records from the right table, and the matched records from the left table.
- **FULL (OUTER) JOIN**: Retrieves all records when there is a match in either the left or right table. *(Note: MySQL does not support `FULL JOIN` directly; it is simulated using a `UNION` of a `LEFT JOIN` and a `RIGHT JOIN`.)*

### Visualizing JOINs with Venn-like Diagrams

The following diagrams illustrate the rows returned by each join type, where the green color indicates the returned dataset.


In [ ]:
# Install matplotlib-venn if not already installed
%pip install matplotlib-venn

#### 1. INNER JOIN

In [ ]:
from matplotlib import pyplot as plt
from matplotlib_venn import venn2

# Create a Venn diagram with 2 circles
v = venn2(subsets=(10, 10, 5), set_labels=("Table A", "Table B"))

# Customize colors for INNER JOIN (only intersection highlighted)
v.get_patch_by_id("10").set_color("white")  # Only A
v.get_patch_by_id("01").set_color("white")  # Only B
v.get_patch_by_id("11").set_color("skyblue")  # Intersection

# Add borders to circles
for circle in ["10", "01", "11"]:
    patch = v.get_patch_by_id(circle)
    if patch:
        patch.set_edgecolor("black")
        patch.set_linewidth(1)

# Hide subset labels
for label in v.subset_labels:
    if label:
        label.set_text("")

plt.title("INNER JOIN")
plt.show()

**Description**: The `INNER JOIN` returns only the rows where there is a match in both tables. If a row in Table A does not have a matching row in Table B, that row is excluded from the result set.

**Use-Case**: This join is used when data is strictly required from both tables. For example, fetching orders with their corresponding customer details.

**Example**:
```sql
SELECT s.name, e.course_id
FROM normalized_students s
INNER JOIN normalized_enrollments e ON s.student_id = e.student_id;
```

#### 2. LEFT JOIN

In [ ]:
from matplotlib import pyplot as plt
from matplotlib_venn import venn2

v = venn2(subsets=(10, 10, 5), set_labels=("Table A", "Table B"))

# Customize colors for LEFT JOIN (Table A and intersection highlighted)
v.get_patch_by_id("10").set_color("skyblue")  # Only A
v.get_patch_by_id("01").set_color("white")  # Only B
v.get_patch_by_id("11").set_color("skyblue")  # Intersection

for circle in ["10", "01", "11"]:
    patch = v.get_patch_by_id(circle)
    if patch:
        patch.set_edgecolor("black")
        patch.set_linewidth(1)

for label in v.subset_labels:
    if label:
        label.set_text("")

plt.title("LEFT JOIN")
plt.show()

**Description**: The `LEFT JOIN` (or `LEFT OUTER JOIN`) returns all rows from the left table (Table A), and the matched rows from the right table (Table B). If no match is found, `NULL` values are returned for the right table's columns.

**Use-Case**: This join is used when all records from the primary (left) table are needed, regardless of whether they have a match in the secondary (right) table. For example, listing all students and their enrolled courses, including students with no enrollments.

**Example**:
```sql
SELECT s.name, e.course_id
FROM normalized_students s
LEFT JOIN normalized_enrollments e ON s.student_id = e.student_id;
```

#### 3. RIGHT JOIN

In [ ]:
from matplotlib import pyplot as plt
from matplotlib_venn import venn2

v = venn2(subsets=(10, 10, 5), set_labels=("Table A", "Table B"))

# Customize colors for RIGHT JOIN (Table B and intersection highlighted)
v.get_patch_by_id("10").set_color("white")  # Only A
v.get_patch_by_id("01").set_color("skyblue")  # Only B
v.get_patch_by_id("11").set_color("skyblue")  # Intersection

for circle in ["10", "01", "11"]:
    patch = v.get_patch_by_id(circle)
    if patch:
        patch.set_edgecolor("black")
        patch.set_linewidth(1)

for label in v.subset_labels:
    if label:
        label.set_text("")

plt.title("RIGHT JOIN")
plt.show()

**Description**: The `RIGHT JOIN` (or `RIGHT OUTER JOIN`) returns all rows from the right table (Table B), and the matched rows from the left table (Table A). If no match is found, `NULL` values are returned for the left table's columns.

**Use-Case**: This join is used when all records from the right table are prioritized. In practice, `LEFT JOIN` is preferred for readability, and `RIGHT JOIN` queries are often rewritten as `LEFT JOIN` by swapping the tables. For example, listing all courses and the students enrolled in them, ensuring courses with no students are still listed.

**Example**:
```sql
SELECT s.name, e.course_id
FROM normalized_students s
RIGHT JOIN normalized_enrollments e ON s.student_id = e.student_id;
```

#### 4. FULL JOIN

In [ ]:
from matplotlib import pyplot as plt
from matplotlib_venn import venn2

v = venn2(subsets=(10, 10, 5), set_labels=("Table A", "Table B"))

# Customize colors for FULL JOIN (All regions highlighted)
v.get_patch_by_id("10").set_color("skyblue")  # Only A
v.get_patch_by_id("01").set_color("skyblue")  # Only B
v.get_patch_by_id("11").set_color("skyblue")  # Intersection

for circle in ["10", "01", "11"]:
    patch = v.get_patch_by_id(circle)
    if patch:
        patch.set_edgecolor("black")
        patch.set_linewidth(1)

for label in v.subset_labels:
    if label:
        label.set_text("")

plt.title("FULL OUTER JOIN")
plt.show()

**Description**: The `FULL JOIN` (or `FULL OUTER JOIN`) returns all rows when there is a match in either the left or right table. It effectively combines the results of both `LEFT JOIN` and `RIGHT JOIN`.

**Use-Case**: This join is used when complete consolidation of data from both tables is required. For example, consolidating user lists from two different systems to identify overlaps and unique entries in both.

**Example**:
```sql
SELECT s.name, e.course_id
FROM normalized_students s
FULL OUTER JOIN normalized_enrollments e ON s.student_id = e.student_id;
```

## Comparison of JOINs

Understanding when to use each type of `JOIN` is critical for writing correct and efficient SQL queries. The following table summarizes the differences and typical use-cases for each join type:

### Summary Table

| JOIN Type | Left Table Rows | Right Table Rows | Match Required? | Common Use-Case |
| :--- | :--- | :--- | :--- | :--- |
| **INNER** | Only matching | Only matching | Yes | Strict intersection |
| **LEFT** | All | Only matching or NULL | No (Left prioritized) | Master list with optional details |
| **RIGHT** | Only matching or NULL | All | No (Right prioritized) | Master list with optional details (rarely used) |
| **FULL** | All | All | No | Complete consolidation |

## The `UNION` Operator: Combining Results

While `JOIN` operations combine columns from different tables horizontally, the `UNION` operator combines rows from different queries vertically.

### What `UNION` Does
The `UNION` operator is used to combine the result sets of two or more `SELECT` statements into a single result set. By default, it removes duplicate rows. To retain all rows including duplicates, `UNION ALL` is used.

### Rules for Using `UNION`
1.  Each `SELECT` statement within the `UNION` must have the same number of columns.
2.  The columns must also have similar data types.
3.  The columns in each `SELECT` statement must be in the same order.

### `UNION` vs. `JOIN`: The Core Difference

*   **JOINs combine data horizontally**: They add columns from another table based on a matching condition.
*   **UNIONs combine data vertically**: They add rows from another result set, effectively appending them.

**Visual Analogy:**
*   **JOIN**: Putting two tables side-by-side and connecting rows with string.
*   **UNION**: Stacking two tables on top of each other to make a taller table.

### Example: `UNION` vs. `FULL JOIN`
A common confusion occurs between `UNION` and `FULL JOIN`.

**Example Scenario**: Consolidating a list of active customers and a list of archived customers.

**Using `UNION` (Vertical Stacking)**:
This query returns a single list of all customer IDs and names, whether active or archived.
```sql
SELECT customer_id, name FROM active_customers
UNION
SELECT customer_id, name FROM archived_customers;
```
*Result*: A single column for `customer_id` and a single column for `name`, with rows from both tables combined.

**Using `FULL JOIN` (Horizontal Alignment)**:
This query attempts to align active and archived customers on their IDs.
```sql
SELECT a.customer_id AS active_id, a.name AS active_name, 
       r.customer_id AS archived_id, r.name AS archived_name
FROM active_customers a
FULL OUTER JOIN archived_customers r ON a.customer_id = r.customer_id;
```
*Result*: Four columns. If a customer exists in both tables, their data appears on the same row. If they exist in only one, the other side has `NULL`s.

### Summary of Use-Cases

| Operation | Direction | Purpose | Typical Scenario |
| :--- | :--- | :--- | :--- |
| **JOIN** | Horizontal | Combining related data from different entities. | Linking orders to customers. |
| **UNION** | Vertical | Consolidating similar data from different sources. | Combining lists of leads and customers. |

## Common Table Expressions (CTE)

A Common Table Expression (CTE) is a temporary result set that can be referenced within a `SELECT`, `INSERT`, `UPDATE`, or `DELETE` statement. It is defined using the `WITH` clause.

### Why CTEs are Needed
CTEs improve the readability and maintainability of complex queries by breaking them down into smaller, logical building blocks. They are particularly useful as an alternative to complex subqueries in the `FROM` clause (derived tables).

### CTE vs. Non-CTE Examples

**Example 1: Using a Subquery in the `FROM` Clause (Non-CTE)**
In this approach, the subquery is nested within the main query, which can become difficult to read as complexity increases.
```sql
SELECT c.course_name, ce.enrollment_count
FROM (
    SELECT course_id, COUNT(student_id) AS enrollment_count
    FROM normalized_enrollments
    GROUP BY course_id
) AS ce
JOIN normalized_courses c ON ce.course_id = c.course_id
WHERE ce.enrollment_count > 5;
```

**Example 2: Using a CTE (Cleaner Alternative)**
The same logic is expressed by defining the temporary result set first, separating the logic from the main query.
```sql
WITH CourseEnrollments AS (
    SELECT course_id, COUNT(student_id) AS enrollment_count
    FROM normalized_enrollments
    GROUP BY course_id
)
SELECT c.course_name, ce.enrollment_count
FROM CourseEnrollments ce
JOIN normalized_courses c ON ce.course_id = c.course_id
WHERE ce.enrollment_count > 5;
```

### Performance Insights
*   **Materialization**: In some database systems (like older versions of PostgreSQL), CTEs are materialized, meaning the database computes the result of the CTE and stores it in memory/disk before running the main query. This can act as an optimization fence, preventing the optimizer from pushing predicates (like `WHERE` conditions) into the CTE.
*   **Modern Optimizers**: Modern database engines (including PostgreSQL 12+ and MySQL 8.0+) often inline CTEs if they are not recursive and not side-effecting, allowing the optimizer to optimize the entire query as a whole, similar to derived tables.
*   **Readability vs. Performance**: While CTEs significantly improve readability, they do not inherently make queries faster than derived tables. In fact, they can sometimes be slower if materialization occurs unnecessarily.

### Leading Practices: When to Use and When Not to Use

**When to Use:**
1.  **Complex Queries**: When a query involves multiple levels of nesting or joins on aggregated data, CTEs make the code much easier to read and debug.
2.  **Code Reusability**: When the same derived dataset needs to be referenced multiple times within the same query.

**When Not to Use:**
1.  **Simple Queries**: For very simple lookups or calculations, adding a CTE adds unnecessary verbosity.
2.  **Performance Bottlenecks**: If a query is slow and execution plan analysis (`EXPLAIN`) reveals that a CTE is being materialized inefficiently, refactoring to a standard subquery or join may be necessary.

## Recursive Queries using CTE

A recursive query is a query that references itself. It is used to traverse hierarchical or graph-structured data, such as organizational hierarchies (manager-employee relationships), file systems (folders and subfolders), or network graphs.

A recursive CTE typically consists of two parts:
1.  **Anchor Member**: The initial query that returns the base result set.
2.  **Recursive Member**: The query that references the CTE itself and is executed repeatedly, joining with the previous result until no new rows are returned.

**Example Use-Case**: Finding all subordinates of a specific manager in an organization chart.

**Example of a Recursive Query:**
```sql
WITH RECURSIVE OrgChart AS (
    -- Anchor Member: Find the top manager
    SELECT employee_id, name, manager_id, 1 AS level
    FROM employees
    WHERE manager_id IS NULL
    
    UNION ALL
    
    -- Recursive Member: Find employees who report to people already in OrgChart
    SELECT e.employee_id, e.name, e.manager_id, o.level + 1
    FROM employees e
    JOIN OrgChart o ON e.manager_id = o.employee_id
)
SELECT * FROM OrgChart;
```

**Explanation of the Example:**
*   **`WITH RECURSIVE OrgChart AS (...)`**: Defines a recursive CTE named `OrgChart`.
*   **Anchor Member**: The first `SELECT` statement retrieves the starting point of the hierarchy (e.g., the top manager where `manager_id IS NULL`). This query runs only once.
*   **`UNION ALL`**: Combines the results of the anchor member and the recursive member.
*   **Recursive Member**: The second `SELECT` statement joins the `employees` table with the `OrgChart` CTE itself. It finds employees whose `manager_id` matches the `employee_id` of someone already identified in the previous iteration. This process repeats until no new rows are returned.
*   **`level + 1`**: Tracks the depth of the hierarchy.

## Subqueries vs. JOINs: A Performance and Readability Comparison

When solving multi-step data retrieval problems, both subqueries and `JOIN` operations can often achieve the same result. However, their performance characteristics and logical implications differ significantly.

### 1. Logic and Readability
*   **Subqueries** can be more intuitive when the problem involves checking for existence or non-existence (e.g., "Find all records NOT IN another set").
*   **JOINs** are generally preferred when data from multiple tables needs to be combined and returned in the final result set, reflecting the relational nature of the database.

### 2. Performance Considerations
*   **Correlated Subqueries**: These reference columns from the outer query and execute once for every row processed by the outer query. This often leads to severe performance degradation on large datasets due to row-by-row processing.
*   **Non-Correlated Subqueries**: Modern database optimizers are highly sophisticated and frequently rewrite non-correlated subqueries (e.g., in the `WHERE` clause) as `JOIN` operations internally.
*   **Indexing**: `JOIN` operations typically leverage indexes on the join conditions more effectively than complex subqueries.

---

### 3. Understanding Semi-Joins

A **Semi-Join** is an optimization technique where the database returns rows from the first table as soon as a matching row is found in the second table, without actually combining the data or continuing to search for multiple matches. It is commonly used for `EXISTS` or `IN` operations.

**Scenario**: Finding all students who have enrolled in at least one course.

**Approach A: Traditional `JOIN` (Potential Duplicates)**
This approach joins the tables. If a student has enrolled in 5 courses, the join produces 5 rows for that student, requiring a `DISTINCT` operation to clean up the result.
```sql
SELECT DISTINCT s.name
FROM normalized_students s
JOIN normalized_enrollments e ON s.student_id = e.student_id;
```

**Approach B: Semi-Join via `EXISTS` (Efficient)**
The database stops searching for a specific student as soon as the *first* matching enrollment is found. No duplicates are generated, making it faster.
```sql
SELECT s.name
FROM normalized_students s
WHERE EXISTS (
    SELECT 1 FROM normalized_enrollments e WHERE e.student_id = s.student_id
);
```
*Learning*: When only the existence of a match is needed (and no columns from the second table are required), a semi-join via `EXISTS` is often more efficient than a traditional `JOIN` with `DISTINCT`.

---

### Conclusive Learning and Guidelines

To determine whether to use a Subquery or a `JOIN`, the following guidelines should be applied:

1.  **Use `JOIN` when**: Columns from both tables are needed in the final `SELECT` output.
2.  **Use Subqueries/`EXISTS` (Semi-Join) when**: Only testing for existence or filtering based on another table is required, without returning its columns.
3.  **Avoid Correlated Subqueries**: If a subquery must be used, ensure it is non-correlated or check if it can be rewritten as a `JOIN` to avoid row-by-row processing.
4.  **Trust but Verify the Optimizer**: For simple queries, the performance difference may be negligible as the database may rewrite the query. However, for complex queries, profiling with `EXPLAIN` is necessary to ensure the most efficient execution plan is used.

## Execution Plan Analysis (`EXPLAIN`)

To optimize queries effectively, one must understand how the database engine plans to execute them. The `EXPLAIN` statement in MySQL provides a detailed breakdown of the query execution plan.

### Key Columns in `EXPLAIN` Output:
- **`type`**: Indicates the join type or access method. This is a critical indicator of performance.
  - `ALL`: Full table scan (worst performance).
  - `index`: Full index scan.
  - `range`: Index range scan.
  - `ref`: Non-unique index lookup.
  - `const`: Unique index lookup (best performance).
- **`possible_keys`**: The indexes that the database could potentially use.
- **`key`**: The index that the database actually decided to use.
- **`rows`**: The estimated number of rows the database needs to examine.
- **`Extra`**: Additional information, such as `Using filesort` or `Using temporary` (both indicate performance overhead).

### Supplemental Insights:
- **Reading `EXPLAIN`**: A value of `ALL` in the `type` column usually signals that an index is missing or not being used, necessitating optimization.
- **Cost-Based Optimizer**: The database optimizer makes decisions based on internal statistics. Sometimes it may choose a full table scan over an index if it estimates that reading the index and then the table would be more expensive (e.g., for small tables or non-selective conditions).
- **Composite Indexes & The Leftmost Prefix Rule**: A frequent evaluation topic involves composite indexes (indexes spanning multiple columns). The database can utilize a composite index on `(col1, col2)` only if the query filters by `col1` alone, or by `col1` AND `col2`. It cannot utilize the index if the query filters only by `col2`. This constraint is known as the Leftmost Prefix Rule.

## The Need for Query Optimization

Before exploring specific optimization techniques, it is essential to understand *why* query optimization is a critical discipline in database management.

### Why Optimize Queries?
1. **Scalability**: As data volume grows, unoptimized queries degrade in performance exponentially. Optimization ensures the system can handle increased load without a linear increase in resource requirements.
2. **Cost Reduction**: In cloud environments, database resources (CPU, memory, I/O) are metered. Inefficient queries consume more resources, leading to higher infrastructure costs.
3. **User Experience**: Slow queries result in slow application responses, directly impacting user satisfaction and engagement.
4. **Resource Availability**: A single slow query can lock resources (tables, rows, memory) and prevent other queries from executing, leading to system-wide bottlenecks.

### Ways Queries Can Be Optimized:
- **Schema Design**: Proper normalization, appropriate data types, and strategic de-normalization.
- **Indexing**: Creating B-Tree or Hash indexes on frequently searched or joined columns.
- **Query Rewriting**: Avoiding `SELECT *`, filtering early, using `LIMIT`, and replacing correlated subqueries with `JOIN`s where appropriate.
- **Database Configuration**: Tuning buffer pool sizes, query cache settings (if applicable), and connection limits.

## Query Optimization Techniques

Applying fundamental query optimization techniques can drastically improve database performance and reduce resource consumption.

### Best Practices:
1. **Avoid `SELECT *`**: Retrieve only the specific columns required. This reduces network payload, memory usage, and allows the database to process queries more effectively.
2. **Filter Early**: Place restrictive conditions in the `WHERE` clause to eliminate rows as early as possible in the execution pipeline.
3. **Use `LIMIT`**: Restrict the result set size when only a subset of rows is needed (e.g., for pagination), reducing the amount of data processed and transmitted.

### Supplemental Insights:
- **Pagination Strategies**: For large datasets, avoid `OFFSET` for deep pagination as it requires scanning and discarding rows. Use keyset pagination (cursor-based) instead.

## Comprehensive Comparison: Database Performance and Scaling Strategies

To achieve high performance and scalability in database systems, various strategies are employed. While some concepts may sound similar, they serve distinct purposes and operate at different layers of the system.

The following table provides a comparison of these concepts, along with typical use-cases.

| Concept | Mechanism | Primary Benefit | Typical Use-Case Example |
| :--- | :--- | :--- | :--- |
| **Indexing** | Creates a physical lookup structure (e.g., B-Tree) pointing to data rows. | Accelerates read operations (`SELECT`). | Searching for a specific customer by email address in a table with millions of rows. |
| **Views** | A virtual table defined by a stored query; executed at runtime. | Simplifies complex queries and restricts access to sensitive columns. | Providing non-technical users with a simplified view of orders without showing full payment details. |
| **Materialized Views** | A physical table that stores the pre-computed result of a query; requires periodic refreshing. | Drastically speeds up complex aggregations by avoiding runtime execution. | Generating daily sales summary reports on platforms processing millions of transactions. |
| **Partitioning** | Divides a single large table into smaller, manageable pieces (partitions) on the *same server*. | Improves maintainability and query performance by scanning only relevant partitions. | Storing logs by date, allowing queries for today's logs to ignore data from previous months. |
| **Sharding** | Distributes data across multiple *separate database servers* (horizontal partitioning). | Enables massive scale beyond the capacity of a single server. | A global social media platform distributing user data across servers based on geographic region. |
| **Replication** | Copies data from a primary database server to one or more secondary servers. | Improves read performance and provides high availability (failover). | A news website directing write traffic to a master database and read traffic to multiple replica databases. |
| **Vertical Scaling** | Increasing the capacity of a single server (adding more CPU, RAM, or storage). | Simple to implement; requires no application-level changes. | Upgrading a database server from 16GB to 64GB RAM to handle increased daily traffic. |
| **Horizontal Scaling** | Adding more servers to the database cluster (often combined with sharding or replication). | Provides theoretical infinite scale and fault tolerance. | Expanding a database cluster from 3 nodes to 10 nodes to handle peak holiday shopping traffic. |

## Advanced Database Operations: Reliability, Maintenance, and Query Impact

Understanding advanced database operations is critical for optimizing query performance and ensuring system reliability. This section explores concepts such as Write-Ahead Logs, data scrubbing, backups, and how they relate to query execution.

### Key Concepts:

- **Write-Ahead Logging (WAL)**: A fundamental reliability mechanism where all changes are written to a sequential log file *before* they are applied to the actual database files. This ensures durability in case of a crash. In MySQL (InnoDB), this is represented by the Redo Log.
- **Data Scrubbing**: A background maintenance process that periodically reads data blocks to detect and correct silent corruption (e.g., bit rot).
- **Data Replication**: The process of copying data across multiple servers. This allows directing read queries to replica servers, reducing the load on the primary server and improving read throughput.
- **Data Dumps**: Logical backups that export database structure and data as SQL statements. Taking a data dump can be resource-intensive and may lock tables, severely impacting active queries.
- **Backups vs. Snapshots**:
  - **Backup**: An independent, complete copy of data stored separately, often requiring time to create and restore.
  - **Snapshot**: A point-in-time state of the storage system, often created near-instantaneously using copy-on-write technology, useful for quick recovery points.
- **Point-in-Time Recovery (PITR)**: The process of restoring a database to a specific timestamp by applying transaction logs (WAL) on top of a full backup.

### Relation to Query Optimization and Analysis:
- **WAL Overhead**: Queries that modify large volumes of data (e.g., massive batch inserts or updates) generate significant WAL traffic, which can become a write bottleneck. Optimizing queries to modify only necessary data reduces this overhead.
- **Replication Lag**: When reading from replicas, queries may retrieve slightly stale data due to replication lag (the time taken for changes to propagate). Queries requiring strict consistency must target the primary server.
- **Resource Contention**: Operations like data dumps consume significant I/O and CPU resources. Running complex analytical queries (subqueries, heavy joins) during these maintenance windows can lead to severe performance degradation.

### Query Profiling with `EXPLAIN ANALYZE`
While standard `EXPLAIN` provides estimates of query execution plans, modern database systems (like MySQL 8.0+) support `EXPLAIN ANALYZE`. This statement actually executes the query and provides a detailed tree representation of the execution steps, along with the actual time taken at each step. This is the standard approach for deep query profiling.

In [ ]:
# Note: While Faker is widely used for generating mock data, libraries like Mimesis
# are often preferred for higher performance when generating massive datasets.
# For the scale of this lab (10,000+ records), Faker is sufficient.

import os
import time
import mysql.connector
from faker import Faker

# An instance of the Faker class is created to generate randomized data.
fake = Faker()


def setup_database():
    """
    Establishes a connection to the MySQL server, creates the target database
    if it does not exist, and returns a connection to that specific database.
    Uses the credentials and database name established in Lecture 1.
    """
    # First connection is made without specifying a database to allow creation.
    conn = mysql.connector.connect(
        host=os.getenv("MYSQL_HOST", "localhost"),
        user="student",
        password=os.getenv("MYSQL_DB_PASSWORD", "root"),
    )
    cursor = conn.cursor()
    # DDL statement to create the database if missing.
    cursor.execute("CREATE DATABASE IF NOT EXISTS student_tasks")
    conn.commit()
    cursor.close()
    conn.close()

    # A new connection is returned, targeting the specific database.
    return mysql.connector.connect(
        host=os.getenv("MYSQL_HOST", "localhost"),
        user="student",
        password=os.getenv("MYSQL_DB_PASSWORD", "root"),
        database="student_tasks",
    )


def setup_schema(conn):
    """
    Creates the necessary tables for the e-commerce simulation.
    Tables are dropped if they exist to ensure a clean state.
    """
    cursor = conn.cursor()

    # Tables are dropped in reverse order of dependencies to avoid foreign key constraint errors.
    cursor.execute("DROP TABLE IF EXISTS orders")
    cursor.execute("DROP TABLE IF EXISTS products")
    cursor.execute("DROP TABLE IF EXISTS users")

    # The Users table stores customer profile information.
    cursor.execute("""
    CREATE TABLE users (
        id INT AUTO_INCREMENT PRIMARY KEY,
        name VARCHAR(100),
        email VARCHAR(100),
        created_at DATETIME
    )
    """)

    # The Products table stores item details.
    cursor.execute("""
    CREATE TABLE products (
        id INT AUTO_INCREMENT PRIMARY KEY,
        name VARCHAR(100),
        price DECIMAL(10, 2)
    )
    """)

    # The Orders table references the Users table via a Foreign Key (user_id).
    cursor.execute("""
    CREATE TABLE orders (
        id INT AUTO_INCREMENT PRIMARY KEY,
        user_id INT,
        total_value DECIMAL(10, 2),
        created_at DATETIME,
        FOREIGN KEY (user_id) REFERENCES users(id)
    )
    """)

    conn.commit()
    cursor.close()


def seed_data(conn):
    """
    Populates the tables with large volumes of mock data to simulate a production environment.
    Batch insertion (executemany) is utilized for performance efficiency.
    """
    cursor = conn.cursor()

    print("Generating users...")
    users_data = []
    # 10,000 user records are generated in memory.
    for _ in range(10000):
        users_data.append((fake.name(), fake.email(), fake.date_time_this_decade()))

    # Batch insertion executes a single operation for all rows, minimizing network roundtrips.
    cursor.executemany(
        "INSERT INTO users (name, email, created_at) VALUES (%s, %s, %s)", users_data
    )

    print("Generating products...")
    products_data = []
    for _ in range(500):
        products_data.append(
            (
                fake.word(),
                round(fake.pydecimal(left_digits=2, right_digits=2, positive=True), 2),
            )
        )

    cursor.executemany(
        "INSERT INTO products (name, price) VALUES (%s, %s)", products_data
    )

    conn.commit()

    # User IDs are fetched to ensure Foreign Key integrity when generating orders.
    cursor.execute("SELECT id FROM users")
    user_ids = [row[0] for row in cursor.fetchall()]

    print("Generating orders...")
    orders_data = []
    for _ in range(20000):
        # A random user ID is selected from the existing users.
        user_id = fake.random_element(elements=user_ids)
        orders_data.append(
            (
                user_id,
                round(fake.pydecimal(left_digits=3, right_digits=2, positive=True), 2),
                fake.date_time_this_decade(),
            )
        )

    cursor.executemany(
        "INSERT INTO orders (user_id, total_value, created_at) VALUES (%s, %s, %s)",
        orders_data,
    )

    conn.commit()
    cursor.close()

In [ ]:
# --- Execution Block ---
# The database is initialized, schema created, and data seeded within a try-except block to handle errors.
try:
    conn = setup_database()
    print("Connected to database.")
    setup_schema(conn)
    print("Schema created.")
    start_time = time.time()
    seed_data(conn)
    print(f"Seeded data in {time.time() - start_time:.2f} seconds.")
    conn.close()
except Exception as e:
    print(f"Error occurred during database setup or seeding: {e}")

In [ ]:
def profile_query(conn, email):
    """
    Runs EXPLAIN ANALYZE on the search query to get actual execution times.
    Note: This requires MySQL 8.0 or later.
    """
    cursor = conn.cursor()
    # EXPLAIN ANALYZE executes the query and prints the plan with timing.
    query = "EXPLAIN ANALYZE SELECT id, name, email FROM users WHERE email = %s"
    try:
        cursor.execute(query, (email,))
        result = cursor.fetchone()
        print("Query Profile (Actual Execution):")
        print(result[0])
    except Exception as e:
        print(f"EXPLAIN ANALYZE failed (likely due to unsupported MySQL version): {e}")
    finally:
        cursor.close()


# Execution
try:
    conn = setup_database()

    # A target email is fetched to ensure availability.
    cursor = conn.cursor()
    cursor.execute("SELECT email FROM users LIMIT 1 OFFSET 5000")
    target_email = cursor.fetchone()[0]
    cursor.close()

    profile_query(conn, target_email)
    conn.close()
except Exception as e:
    print(f"Error occurred: {e}")

## Note on Production vs. Educational Coding Practices

The code examples provided in this laboratory are designed for educational clarity and self-paced learning. Consequently, certain simplifications have been made that deviate from standard production practices.

### Key Differences:
- **Error Handling**: The examples utilize broad `try...except` blocks or minimal exception handling to ensure students see error messages directly. In production systems, specific exceptions (e.g., `mysql.connector.Error`) should be caught and handled gracefully.
- **Logging**: Standard `print()` statements are used for immediate feedback in the notebook. Production systems utilize formal logging libraries (e.g., Python's `logging`) to manage log levels (INFO, WARN, ERROR) and output destinations.
- **Connection Management**: Connections are opened and closed directly to show the flow of operations. Production environments utilize connection pooling to reuse active connections and reduce latency.
- **Hardcoded Defaults**: While environment variables are supported, fallback defaults are provided in the code for ease of use by students. Production systems strictly avoid hardcoding defaults for security sensitive parameters.

## Hands-On Lab: E-Commerce Data Analyzer

### Objective
This laboratory exercise simulates the role of a backend data engineer for an e-commerce platform. The goal is to write complex analytical queries, analyze their performance, identify bottlenecks, and apply optimization techniques (indexing) to improve execution times.

### Hands-On Lab Part 3: Dataset Seeding
To simulate a realistic environment where query performance matters, a large volume of mock data will be generated and inserted into the database using the `Faker` library.

*Note: The Faker library was installed in a previous step.*

In [ ]:
# The following code defines a Python decorator.
# Decorators are used to modify or enhance the behavior of functions without changing their source code.
# Here, the decorator is used to automatically measure and log the execution time of database queries.

import time
import functools


def time_query(func):
    """
    A decorator that measures and prints the execution time of a function.
    It wraps the target function, records the start and end times, and prints the difference.
    """

    # functools.wraps is used to preserve the original function's metadata (like its name and docstring).
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # The start time is recorded before calling the function.
        start_time = time.time()

        # The actual function is executed here, passing any arguments it received.
        result = func(*args, **kwargs)

        # The end time is recorded after the function completes.
        end_time = time.time()

        # The execution time is calculated and printed to the console.
        print(
            f"Function '{func.__name__}' executed in {end_time - start_time:.4f} seconds."
        )

        # The result of the original function is returned.
        return result

    # The wrapper function is returned, effectively replacing the original function.
    return wrapper

### Hands-On Lab Part 4: Implementing Subqueries

In this phase, a reporting function is implemented to identify high-value customers. A high-value customer is defined as a user whose total order value is greater than the average order value of all orders in the platform. This requires a subquery to calculate the average order value first.

In [ ]:
# The @time_query decorator is applied to automatically measure the execution time of this function.
@time_query
def get_high_value_customers(conn):
    """
    Retrieves users whose total order value is greater than the average order value.
    Demonstrates the use of a subquery within the HAVING clause.
    """
    cursor = conn.cursor()

    # The query joins the users and orders tables, groups the results by user,
    # and filters groups using a subquery in the HAVING clause.
    query = """
    SELECT u.id, u.name, SUM(o.total_value) AS user_total
    FROM users u
    JOIN orders o ON u.id = o.user_id
    GROUP BY u.id, u.name
    -- The subquery below calculates the average value of all orders in the platform.
    HAVING SUM(o.total_value) > (
        SELECT AVG(total_value) FROM orders
    )
    ORDER BY user_total DESC
    LIMIT 10;
    """

    cursor.execute(query)
    results = cursor.fetchall()
    cursor.close()
    return results


# --- Execution Block ---
try:
    conn = setup_database()
    print("Fetching high-value customers...")
    # The decorator will print the execution time automatically.
    customers = get_high_value_customers(conn)
    for c in customers:
        print(f"User ID: {c[0]}, Name: {c[1]}, Total Spent: {c[2]}")
    conn.close()
except Exception as e:
    print(f"Error occurred: {e}")

### Hands-On Lab Part 5: Identifying the Performance Bottleneck

In this phase, a deliberately unoptimized query is executed to demonstrate performance issues on non-indexed columns. The query searches for a specific user by their email address in the `users` table. Since no index has been created on the `email` column yet, the database must perform a full table scan.

The execution time is recorded using Python's `time` module.

In [ ]:
# The @time_query decorator is applied to automatically measure the execution time of this function.
@time_query
def find_user_by_email(conn, email):
    """
    Searches for a user by email.
    Demonstrates performance without an index.
    """
    cursor = conn.cursor()
    # A standard SELECT query with a WHERE clause filtering on a non-indexed column.
    query = "SELECT id, name, email FROM users WHERE email = %s"
    cursor.execute(query, (email,))
    result = cursor.fetchone()
    cursor.close()
    return result


# --- Execution Block ---
try:
    conn = setup_database()

    # A target email is fetched from the middle of the table (using OFFSET)
    # to ensure the database must scan a significant number of rows before finding the target.
    cursor = conn.cursor()
    cursor.execute("SELECT email FROM users LIMIT 1 OFFSET 5000")
    target_email = cursor.fetchone()[0]
    cursor.close()

    print(f"Searching for email: {target_email}")

    # The search function is called, and the decorator will print the time taken.
    user = find_user_by_email(conn, target_email)
    print(f"User found: {user}")

    conn.close()
except Exception as e:
    print(f"Error occurred: {e}")

### Hands-On Lab Part 6: Execution Plan Analysis

To verify why the query is slow, the `EXPLAIN` command is executed on the same query. This allows students to see the access method chosen by the database engine.

In [ ]:
def analyze_query(conn, email):
    """
    Runs the EXPLAIN statement on the search query to retrieve the execution plan.
    This helps understand how the database accesses the data.
    """
    cursor = conn.cursor()
    # The EXPLAIN keyword is prepended to the query.
    query = "EXPLAIN SELECT id, name, email FROM users WHERE email = %s"
    cursor.execute(query, (email,))
    results = cursor.fetchall()

    # Column names are fetched from the cursor description to create a readable table.
    columns = [desc[0] for desc in cursor.description]

    cursor.close()
    return columns, results


# --- Execution Block ---
try:
    conn = setup_database()
    # The query is analyzed using the target email fetched in the previous cell.
    columns, plan = analyze_query(conn, target_email)

    print("Execution Plan:")
    # A header row is printed for clarity.
    print(" | ".join(columns))
    print("-" * 80)
    # The execution plan rows are printed. Students should look for the 'type' and 'rows' columns.
    for row in plan:
        print(" | ".join(str(val) for val in row))

    conn.close()
except Exception as e:
    print(f"Error occurred: {e}")